# Research analysis workspace

Load Stata `.dta` files, explore, define exposures, and transform variables. Put data files in `data/raw/` (or update `DATA_DIR` below).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat

# Project root = parent of notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

## 1. Load data

Set `DTA_PATH` to your file. `pyreadstat` returns a DataFrame plus metadata (variable labels, value labels).

In [ ]:
# Example: DTA_PATH = DATA_DIR / "your_file.dta"
DTA_PATH = None  # set this before running

if DTA_PATH is None or not Path(DTA_PATH).is_file():
    dta_files = sorted(DATA_DIR.glob("*.dta"))
    if dta_files:
        DTA_PATH = dta_files[0]
        print(f"Using first .dta in data/raw: {DTA_PATH.name}")
    else:
        raise FileNotFoundError(
            f"No .dta in {DATA_DIR}. Add a file or set DTA_PATH."
        )

df, meta = pyreadstat.read_dta(DTA_PATH)
print(df.shape)
df.head()

## 2. Analysis — inspect structure

Types, missingness, simple summaries.

In [ ]:
df.info()
df.describe(include="all").T.head(20)

## 3. Exposure analysis

Define exposure column(s), check distributions, crosstabs with outcomes, or plots. Replace names with your variables.

In [ ]:
# Example placeholders — replace with your column names
EXPOSURE_COL = None  # e.g. "exposure" or "treatment"
OUTCOME_COL = None  # optional, for stratified views

if EXPOSURE_COL and EXPOSURE_COL in df.columns:
    s = df[EXPOSURE_COL]
    print(s.value_counts(dropna=False))
    fig, ax = plt.subplots(figsize=(6, 3))
    s.astype(str).value_counts().head(15).plot(kind="barh", ax=ax)
    ax.set_title(f"Exposure: {EXPOSURE_COL}")
    plt.tight_layout()
    plt.show()

    if OUTCOME_COL and OUTCOME_COL in df.columns:
        ct = pd.crosstab(df[EXPOSURE_COL], df[OUTCOME_COL], margins=True)
        display(ct)
else:
    print("Set EXPOSURE_COL (and optionally OUTCOME_COL) to run exposure summaries.")

## 4. Transformations

Derived variables, recodes, logs, standardization — keep a clean `df_analysis` copy when chaining steps.

In [ ]:
df_analysis = df.copy()

# Examples (uncomment and adapt):
# df_analysis["log_income"] = np.log(df_analysis["income"].clip(lower=1))
# df_analysis["age_centered"] = df_analysis["age"] - df_analysis["age"].mean()
# df_analysis["high_dose"] = (df_analysis["dose"] > df_analysis["dose"].median()).astype(int)

df_analysis.head()